In [72]:
import pandas as pd
import pyarrow as pa
import pyarrow.csv as ar_csv
# Loading file
# Type in filename below
filename = "kmdc_7_13_26"
print(f"Loading file {filename}.csv...")
df = pd.read_csv(f"{filename}.csv", engine='pyarrow')
print(f"{filename}.csv loaded")
df.columns = pd.io.common.dedup_names(df.columns, is_potential_multiindex=False)

Loading file kmdc_7_13_26.csv...
kmdc_7_13_26.csv loaded


In [74]:
# Setting up variables to be used in partitioning rows and columns
if "kmdc_7_13_26" in filename:
    koi_name = "kmdc_index"
elif "final_kdc" in filename:
    koi_name = "KIC"
else:
    # This is merely a failsafe to prevent file corruption
    print("Unknown file. Edit the python script to read this. Make sure to load the file before running this part.")
    raise SystemExit

df_wout_col = df.drop(koi_name, axis=1)
kmdc = df[koi_name].astype(str)

# ==================================================================================================================
# COLUMN SECTION
# Enter the names EXACTLY as they appear in the files as a string.
# Write the column names with ',' between them. The koi_name column is included automatically.
# Or type 'all' to include every column
inpcol = "all"
# ==================================================================================================================

if inpcol.strip().lower() == "all":
    col = [koi_name] + list(df_wout_col.columns)
else:
    requested_cols = [c.strip() for c in inpcol.split(',')]
    invalid = [x for x in requested_cols if x not in df_wout_col.columns]
    if invalid:
        print(f"Invalid column(s): {', '.join(invalid)}")
        print(f"Your options are: {', '.join(df_wout_col.columns)}")
        raise SystemExit
    col = [koi_name] + requested_cols

# ===========================================================================================
# ROW (KOI) SECTION
# Do you want every KOI? Indicate 'yes' or 'no' below
koi_check = "y"
# If 'no' above, enter the KOI number(s) below (e.g. 456, 992, 1824):
inpkois = "123"
# ===========================================================================================

allrows = koi_check.strip().lower() in ('y', 'yes')

if allrows:
    output_df = df[col]
    valid = []
else:
    kois = [k.strip() for k in inpkois.split(",")]
    valid = [k for k in kois if kmdc.str.startswith(k).any()]
    invalid = [k for k in kois if k not in valid]
 
    if invalid:
        print("No KOIs found beginning with:")
        print(", ".join(invalid))
        raise SystemExit
 
    output_df = df[kmdc.str.startswith(tuple(valid))][col]

# Final confirmation before creating file
print("We will make a .csv file with:")
print("Columns:",col)
if allrows:
    print("KOI: All")
else:
    print("KOIs:", ", ".join(valid))


# ==========================================================================================
# OUTPUT FILE NAME
# What would you like to call the output file? Indicate below.
filename_out = "output_test"
# ==========================================================================================

table = pa.Table.from_pandas(output_df)
ar_csv.write_csv(table, f"{filename_out}.csv")
print(f"Saved {filename_out}.csv")

We will make a .csv file with:
Columns: ['kmdc_index', '', '.1', 'M_s', 'R_s', 'c_1', 'c_2', 'R_p/R_s', 'R_pJ', 'R_pE', 'M_pE', 'rho_p', 'rho_s', 'M_p/M_s', 'M_pJ', 'sqrt(e)_cos(omega)', 'sqrt(e)_sin(omega)', 'i', 'Omega', 'e', 'omega', 'true_anomaly', 'eccentric_anomaly', 'mean_anomaly', 'mean_longitude', 'a_AU', 'a_R_s', 'peri_AU', 'peri_R_s', 'apo_AU', 'apo_R_s', 'd_AU', 'd_R_s', 'Period_days', 'T_0', 'b_trans', 'b_occ', 'p_trans', 'p_occ', 'T_total_hr', 'T_full_hr', 'K_RV', 'occurrence_rate_hsu', 'E_or_hsu', 'e_or_hsu', 'hsu_flag', 'multiplicity', 'P/Pin', 'P/Pout', 'Tdur/Tdurin', 'Tdur/Tdurout', 'R/Rin', 'R/Rout', 'M/Min', 'M/Mout', 'rho/rhoin', 'rho/rhoout', 'i-iin', 'iout-i', 'xiin', 'xiout', 'distin_hillrad', 'distout_hillrad', 'distin_hillrad_e', 'distout_hillrad_e', 'e/ein', 'eout/e', 'omega-omegain', 'omegaout-omega', 'dilute', 'chisq', 'Chain#', 'chisq_rank', 'step_number', 'phodymm_index', 'planet', 'KIC', 'KOI', 'Kepler', 'Period_days_rowe', 'e_Period_rowe', 'T0_rowe', 'e